# 1-Year Sharpe Ratio Comparison

In this notebook, we compare the **annualized Sharpe Ratio** of a selected stock (**TICKER**) 
against the **S&P 500** over the past year.

The Sharpe Ratio measures **risk-adjusted returns**, and allows quick comparison of performance vs. volatility.


In [ ]:
# --- Imports ---
from src.technical_setup import *

## Function: `annualized_sharpe_ratio`

This function calculates the **annualized Sharpe Ratio** for a given price series.

In [ ]:
def annualized_sharpe_ratio(prices, trading_days=252):
    """
    Calculates the annualized Sharpe Ratio for a price series.

    Parameters
    ----------
    prices : pd.Series
        Historical closing prices.
    trading_days : int, optional
        Number of trading days in a year (default 252).

    Returns
    -------
    float
        Annualized Sharpe Ratio.
    """
    returns = prices.pct_change().dropna()
    sharpe = (returns.mean() / returns.std()) * (trading_days**0.5)
    return sharpe


## Function: `plot_sharpe_comparison`

This function plots a **horizontal bar chart** comparing Sharpe Ratios for multiple assets.

In [ ]:
def plot_sharpe_comparison(sharpe_dict, save_path=None):
    """
    Plots a horizontal bar chart of Sharpe Ratios.

    Parameters
    ----------
    sharpe_dict : dict
        Dictionary with asset names as keys and Sharpe Ratios as values.
    save_path : str, optional
        Path to save the figure.
    """
    df_sharpe = pd.DataFrame.from_dict(sharpe_dict, orient='index', columns=['Sharpe Ratio'])
    df_sharpe = df_sharpe.sort_values(by='Sharpe Ratio')

    plt.figure(figsize=(10, 6))
    bars = plt.barh(
        df_sharpe.index,
        df_sharpe['Sharpe Ratio'],
        color=["green" if x > 0 else "red" for x in df_sharpe['Sharpe Ratio']]
    )
    plt.xlabel("Sharpe Ratio")
    plt.title("1-Year Sharpe Ratio Comparison")
    plt.grid(axis="x", linestyle="--", alpha=0.5)

    # Values next to bars
    for i, v in enumerate(df_sharpe['Sharpe Ratio']):
        plt.text(v + 0.01, i, f"{v:.2f}")

    # Save figure if requested
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()


## Example Usage

We now calculate the Sharpe Ratios for **TICKER** and **S&P 500** and plot them.


In [ ]:
# --- Download data ---
df = yf.download([TICKER, "^GSPC"], period="1y")["Close"]

# Handle MultiIndex if returned
if isinstance(df.columns, pd.MultiIndex):
    df_asset = df[TICKER]
    df_sp500 = df["^GSPC"]
else:
    df_asset = df[TICKER]
    df_sp500 = df["^GSPC"]

# --- Calculate Sharpe Ratios ---
sharpe_dict = {
    TICKER: annualized_sharpe_ratio(df_asset),
    "S&P 500": annualized_sharpe_ratio(df_sp500)
}

# --- Plot and save ---
plot_sharpe_comparison(sharpe_dict, save_path=f"figures/{TICKER}_sharpe_ratio.png")
